<a href="https://colab.research.google.com/github/shyampandey263/bolo-summarize/blob/main/bolo_summarize.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bolo Summarize: Hindi voice notes to structured summary

In [ ]:
!pip install -q "nemo-toolkit[asr]" huggingface_hub langchain langchain-anthropic

In [ ]:
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get("HF_TOKEN"))

In [ ]:
import torch
import nemo.collections.asr as nemo_asr

asr_model = nemo_asr.models.ASRModel.from_pretrained("adalat-ai/koyal-hi-120m-1.0")
asr_model = asr_model.to("cuda" if torch.cuda.is_available() else "cpu")
asr_model.eval()

print("Model loaded on:", next(asr_model.parameters()).device)

In [ ]:
from typing import List
from pydantic import BaseModel, Field

class ActionItem(BaseModel):
    task: str
    owner: str = Field(description="Person responsible, or 'Not mentioned'")
    deadline: str = Field(description="Deadline, or 'Not mentioned'")

class NoteSummary(BaseModel):
    title: str
    summary: str = Field(description="3-5 sentence summary")
    key_points: List[str]
    action_items: List[ActionItem]
    numbers_and_dates: List[str] = Field(description="Important amounts, dates, times, counts mentioned")

In [ ]:
!pip install -q langchain-openai

In [ ]:
import os
from google.colab import userdata
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You process Hindi speech transcripts produced by an ASR system. "
     "The transcript may contain recognition errors, so use context to interpret it. "
     "Do not invent facts that are not in the transcript. "
     "Write the output in {language}."),
    ("human", "Transcript:\n\n{transcript}")
])

chain = prompt | llm.with_structured_output(NoteSummary)
print("Chain ready")

In [ ]:
!pip install -q gradio

In [ ]:
import os, glob, subprocess, shutil, json

def process_audio(audio_path, language="English"):
    if audio_path is None:
        return "Please upload an audio file.", "", ""

    # 1. Convert to 16 kHz mono WAV in 30-second chunks
    shutil.rmtree("chunks", ignore_errors=True)
    os.makedirs("chunks", exist_ok=True)
    subprocess.run([
        "ffmpeg", "-y", "-i", audio_path,
        "-ar", "16000", "-ac", "1",
        "-f", "segment", "-segment_time", "30",
        "chunks/chunk_%03d.wav"
    ], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    chunk_files = sorted(glob.glob("chunks/*.wav"))

    # 2. Transcribe with Koyal
    out = asr_model.transcribe(chunk_files, batch_size=2)
    if isinstance(out, tuple):
        out = out[0]
    texts = [h.text if hasattr(h, "text") else str(h) for h in out]
    transcript = " ".join(texts)

    # 3. Summarise with the LLM (Large Language Model)
    res = chain.invoke({"transcript": transcript, "language": language})

    # 4. Format the result
    lines = [f"TITLE: {res.title}", "", f"SUMMARY: {res.summary}", "", "KEY POINTS:"]
    lines += [f"- {p}" for p in res.key_points]
    lines += ["", "ACTION ITEMS:"]
    lines += [f"- {a.task} | Owner: {a.owner} | Deadline: {a.deadline}" for a in res.action_items]
    lines += ["", "NUMBERS & DATES:"]
    lines += [f"- {n}" for n in res.numbers_and_dates]

    return transcript, "\n".join(lines), json.dumps(res.model_dump(), ensure_ascii=False, indent=2)

print("process_audio ready")

In [ ]:
import gradio as gr
import subprocess, json, os

MAX_SECONDS = 300
ALLOWED = [".mp3", ".wav", ".aac", ".m4a", ".ogg", ".flac", ".opus", ".wma"]

def get_duration(path):
    try:
        out = subprocess.check_output(
            ["ffprobe", "-v", "error", "-show_entries", "format=duration",
             "-of", "default=nw=1:nk=1", path]).decode().strip()
        return float(out)
    except Exception:
        return None  # raw .aac files sometimes have no duration info

def safe_process(file_in, mic_in, language):
    try:
        path = file_in if file_in else mic_in
        if path is None:
            return "Please upload or record an audio file.", "", ""
        if not isinstance(path, str):
            path = path.name

        ext = os.path.splitext(path)[1].lower()
        if ext and ext not in ALLOWED:
            return f"Unsupported format {ext}. Use: {', '.join(ALLOWED)}", "", ""

        dur = get_duration(path)
        if dur is not None and dur > MAX_SECONDS:
            return f"Audio is too long ({int(dur)}s). Maximum is {MAX_SECONDS}s.", "", ""

        # Hard-cap the length even when duration is unknown (protects your API credits)
        trimmed = "input_trimmed.wav"
        subprocess.run(["ffmpeg", "-y", "-i", path, "-t", str(MAX_SECONDS),
                        "-ar", "16000", "-ac", "1", trimmed],
                       check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

        return process_audio(trimmed, language)
    except Exception as e:
        return f"Error: {e}", "", ""

with gr.Blocks(title="Bolo Summarize") as demo:
    gr.Markdown("# 🎙️ Bolo Summarize\nUpload Hindi speech (MP3, AAC, WAV, M4A, OGG, FLAC) and get a transcript, summary, and action items.")
    file_in = gr.File(label="Upload audio file (max 5 min)", file_types=ALLOWED, type="filepath")
    mic_in = gr.Audio(sources=["microphone"], type="filepath", label="...or record with microphone")
    lang = gr.Radio(["English", "Hindi"], value="English", label="Summary language")
    btn = gr.Button("Transcribe & Summarize", variant="primary")
    transcript_out = gr.Textbox(label="Hindi transcript (Koyal ASR)", lines=8)
    summary_out = gr.Textbox(label="Structured summary", lines=16)
    json_out = gr.Code(label="JSON output", language="json")
    btn.click(safe_process, [file_in, mic_in, lang], [transcript_out, summary_out, json_out])

demo.launch(share=True, debug=True)

# 📦 Structuring Repository Files for GitHub
Below, we create the standard folder layout for your project and pack it into a ZIP archive for simple GitHub uploading.

In [ ]:
import os
import shutil

# Create clean project directory
repo_dir = 'bolo-summarize'
shutil.rmtree(repo_dir, ignore_errors=True)
os.makedirs(repo_dir, exist_ok=True)
print(f'Created empty repository structure folder: {repo_dir}/')

In [ ]:
%%writefile bolo-summarize/requirements.txt
nemo-toolkit[asr]
huggingface_hub
langchain
langchain-openai
gradio
pydantic
torch

In [ ]:
%%writefile bolo-summarize/app.py
import os
import glob
import subprocess
import shutil
import json
from typing import List
import torch
import gradio as gr
from pydantic import BaseModel, Field
from huggingface_hub import login
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
import nemo.collections.asr as nemo_asr

# Secure credentials login using environment variables
HF_TOKEN = os.getenv("HF_TOKEN")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if HF_TOKEN:
    login(token=HF_TOKEN)

# Load Model
device = "cuda" if torch.cuda.is_available() else "cpu"
asr_model = nemo_asr.models.ASRModel.from_pretrained("adalat-ai/koyal-hi-120m-1.0")
asr_model = asr_model.to(device)
asr_model.eval()

# Define Pydantic response models
class ActionItem(BaseModel):
    task: str
    owner: str = Field(description="Person responsible, or 'Not mentioned'")
    deadline: str = Field(description="Deadline, or 'Not mentioned'")

class NoteSummary(BaseModel):
    title: str
    summary: str = Field(description="3-5 sentence summary")
    key_points: List[str]
    action_items: List[ActionItem]
    numbers_and_dates: List[str] = Field(description="Important amounts, dates, times, counts mentioned")

# Build Chain
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You process Hindi speech transcripts produced by an ASR system. "
     "The transcript may contain recognition errors, so use context to interpret it. "
     "Do not invent facts that are not in the transcript. "
     "Write the output in {language}."),
    ("human", "Transcript:\n\n{transcript}")
])
chain = prompt | llm.with_structured_output(NoteSummary)

MAX_SECONDS = 300
ALLOWED = [".mp3", ".wav", ".aac", ".m4a", ".ogg", ".flac", ".opus", ".wma"]

def get_duration(path):
    try:
        out = subprocess.check_output([
            "ffprobe", "-v", "error", "-show_entries", "format=duration",
            "-of", "default=nw=1:nk=1", path
        ]).decode().strip()
        return float(out)
    except Exception:
        return None

def process_audio(audio_path, language="English"):
    shutil.rmtree("chunks", ignore_errors=True)
    os.makedirs("chunks", exist_ok=True)
    subprocess.run([
        "ffmpeg", "-y", "-i", audio_path,
        "-ar", "16000", "-ac", "1",
        "-f", "segment", "-segment_time", "30",
        "chunks/chunk_%03d.wav"
    ], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    chunk_files = sorted(glob.glob("chunks/*.wav"))

    out = asr_model.transcribe(chunk_files, batch_size=2)
    if isinstance(out, tuple):
        out = out[0]
    texts = [h.text if hasattr(h, "text") else str(h) for h in out]
    transcript = " ".join(texts)

    res = chain.invoke({"transcript": transcript, "language": language})

    lines = [f"TITLE: {res.title}", "", f"SUMMARY: {res.summary}", "", "KEY POINTS:"]
    lines += [f"- {p}" for p in res.key_points]
    lines += ["", "ACTION ITEMS:"]
    lines += [f"- {a.task} | Owner: {a.owner} | Deadline: {a.deadline}" for a in res.action_items]
    lines += ["", "NUMBERS & DATES:"]
    lines += [f"- {n}" for n in res.numbers_and_dates]

    return transcript, "\n".join(lines), json.dumps(res.model_dump(), ensure_ascii=False, indent=2)

def safe_process(file_in, mic_in, language):
    try:
        path = file_in if file_in else mic_in
        if path is None:
            return "Please upload or record an audio file.", "", ""
        if not isinstance(path, str):
            path = path.name

        ext = os.path.splitext(path)[1].lower()
        if ext and ext not in ALLOWED:
            return f"Unsupported format {ext}. Use: {', '.join(ALLOWED)}", "", ""

        dur = get_duration(path)
        if dur is not None and dur > MAX_SECONDS:
            return f"Audio is too long ({int(dur)}s). Maximum is {MAX_SECONDS}s.", "", ""

        trimmed = "input_trimmed.wav"
        subprocess.run([
            "ffmpeg", "-y", "-i", path, "-t", str(MAX_SECONDS),
            "-ar", "16000", "-ac", "1", trimmed
        ], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

        return process_audio(trimmed, language)
    except Exception as e:
        return f"Error: {e}", "", ""

with gr.Blocks(title="Bolo Summarize") as demo:
    gr.Markdown("# 🎙️ Bolo Summarize\nUpload Hindi speech and get structure, transcripts, summaries, and action items instantly.")
    file_in = gr.File(label="Upload audio file (max 5 min)", file_types=ALLOWED, type="filepath")
    mic_in = gr.Audio(sources=["microphone"], type="filepath", label="...or record with microphone")
    lang = gr.Radio(["English", "Hindi"], value="English", label="Summary language")
    btn = gr.Button("Transcribe & Summarize", variant="primary")
    transcript_out = gr.Textbox(label="Hindi transcript (Koyal ASR)", lines=8)
    summary_out = gr.Textbox(label="Structured summary", lines=16)
    json_out = gr.Code(label="JSON output", language="json")
    btn.click(safe_process, [file_in, mic_in, lang], [transcript_out, summary_out, json_out])

if __name__ == "__main__":
    demo.launch()

In [ ]:
%%writefile bolo-summarize/README.md
# 🎙️ Bolo Summarize

Bolo Summarize is a smart speech-to-structured-summary tool tailored for **Hindi Audio**. It utilizes **Koyal ASR** to create highly accurate transcripts of speech and feeds them to **GPT-4o-mini** via LangChain to produce a detailed list of summaries, key discussion points, dates, and actionable tasks mapped to owners.

## 🚀 Project Features
- **Automatic Hindi Transcription**: Handled locally via `adalat-ai/koyal-hi-120m-1.0` model.
- **Structured Summary Layout**: Extracts structured insights, deadlines, assigned tasks, and values using Pydantic.
- **Modern Gradio App**: Easy drag-and-drop or microphone recorder.
- **Multilingual Support**: Generate output in either Hindi or English.

## ⚙️ Running Locally

Ensure you have `ffmpeg` installed on your host system.

1. Install the required Python libraries:
```bash
pip install -r requirements.txt
```

2. Set up your environments keys:
```bash
export HF_TOKEN="your_huggingface_write_token"
export OPENAI_API_KEY="your_openai_api_key"
```

3. Start the application:
```bash
python app.py

In [ ]:
%%writefile bolo-summarize/.gitignore
__pycache__/
*.py[cod]
chunks/
input_trimmed.wav
transcript.txt
summary.json
.env

In [ ]:
shutil.make_archive('bolo-summarize-repo', 'zip', 'bolo-summarize')
print('Generated zip file "bolo-summarize-repo.zip" successfully! Please download it from the file explorer on the left.')